# Credit Risk Modeling

## Setup
Import the libraries we'll need.

In [ ]:
from numpy import argmax, array, concatenate, log, r_, sum, where
from scipy.special import logit
from pandas import DataFrame
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelBinarizer, LabelEncoder, OneHotEncoder, PolynomialFeatures, QuantileTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import auc, confusion_matrix, roc_curve
from seaborn import barplot, heatmap, histplot, lineplot
from matplotlib.pyplot import subplots, tight_layout, show

## Task 1: Data exploration
- Load the German Credit dataset (OpenML data_id=31)
- Inspect it and visualize every variable

In [ ]:
credit_risk = fetch_openml(data_id=31, as_frame=True, parser="auto").frame

In [ ]:
credit_risk.info()

In [ ]:
n_cols = 4
n_rows = -(-len(credit_risk.columns) // n_cols)  # ceiling division

fig, axes = subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
axes = axes.flatten()

for ax, col in zip(axes, credit_risk.columns):
    if credit_risk[col].dtype.name == 'category':
        credit_risk[col].value_counts().plot.bar(ax=ax, color='skyblue', edgecolor='black')
    else:
        credit_risk[col].plot.hist(ax=ax, color='salmon', edgecolor='black')
    ax.set_title(col, fontsize=9)

for ax in axes[len(credit_risk.columns):]:
    ax.set_visible(False)

tight_layout()
show()

## Task 2: Data preparation
- Split into features (z) and target (x)
- Encode the target (good/bad -> 0/1)
- Separate continuous vs categorical features
- One-hot encode categoricals, standardize continuous
- Recombine into one feature matrix z
- Split into estimation (train) and test sets

In [ ]:
raw_target = credit_risk.iloc[:, -1]
features = credit_risk.iloc[:, :-1]

lb = LabelEncoder()
target = lb.fit_transform(array(raw_target))
print(lb.classes_)

LabelEncoder maps the target to integers alphabetically: 'bad' -> 0, 'good' -> 1.

Class 1 = good, so from here on, predicted probability of class 1 means probability this applicant is a good credit risk, not a defaulter.

In [ ]:
i_cat, i_cont = [], []
for i, col in enumerate(features.columns):
    i_cat.append(i) if features[col].dtype.name == 'category' else i_cont.append(i)

features_cat_raw = features.iloc[:, i_cat]
features_cont_raw = features.iloc[:, i_cont]

In [ ]:
features_cat_raw

In [ ]:
features_cont_raw

One-hot encoding turns category labels into a bank of 0/1 switches, so the model can weight each category individually.
QuantileTransformer reshapes skewed numeric columns into a well-behaved, comparable scale, so no single numeric feature distorts the fit just because of its raw magnitude.

In [ ]:
enc = OneHotEncoder(categories='auto', sparse_output=False)
features_cat = enc.fit_transform(features_cat_raw)

In [ ]:
quantile_transformer = QuantileTransformer(output_distribution='normal')
features_cont = quantile_transformer.fit_transform(features_cont_raw)

In [ ]:
print(features_cont.shape)
print(features_cat.shape)

Step D: recombine and split into train/test.

In [ ]:
features_concat = concatenate((features_cont, features_cat), axis=1)

features_train, features_test, target_train, target_test = train_test_split(features_concat, target)

In [ ]:
print(features_train.shape, features_test.shape, target_train.shape, target_test.shape)

## Task 3: Model estimation
- Fit a logistic regression model with L1 (lasso) regularization on the estimation set
- Get predicted probabilities
- Compute ROC curve + AUC on the estimation set

In [ ]:
lambda_lasso = 3
model = LogisticRegression(
    l1_ratio=1, C=1/lambda_lasso,
    solver='liblinear', max_iter=500)
model = model.fit(features_train, target_train)

Predicted probabilities on the training set.

In [ ]:
p_train = model.predict_proba(features_train)[:, 1]

ROC curve and AUC, in-sample.

In [ ]:
fpr, tpr, thresholds = roc_curve(target_train, p_train)
auc_val = auc(fpr, tpr)

roc_curve slides the cutoff across every possible threshold and, at each one, computes the false positive rate and true positive rate.
auc collapses that whole curve into a single summary number: bigger means better separation between good and bad applicants.

In [ ]:
print(auc_val)

## Task 4: Point prediction
- Find the statistically optimal cutoff (Youden's J)
- Choose a business cutoff (more conservative)
- Build point predictions using the cutoff
- Compute confusion matrix

Step A: the statistically optimal cutoff (Youden's J). It finds the cutoff that maximizes tpr minus fpr, i.e. the point on the ROC curve farthest above the useless diagonal line.

In [ ]:
optimal_idx = argmax(tpr - fpr)
stat_cutoff = thresholds[optimal_idx]

In [ ]:
stat_cutoff

Step B: the business cutoff.

A false positive (predicting good when the applicant is actually bad) means the bank loses money on a defaulted loan, which is much worse than a false negative (rejecting someone who would have paid it back fine). So a business wants a more conservative (higher) cutoff than the equally weighted statistical optimum, demanding a high predicted probability of good before approving.

In [ ]:
business_cutoff = 0.72

Step C: build point predictions and a confusion matrix.

In [ ]:
x_bar = (p_train >= business_cutoff).astype(int)
cm = confusion_matrix(target_train, x_bar, normalize='all')
cm

False positive rate is about 5.2 percent: the bank approves a loan to a truly bad-risk applicant only rarely, which is what a conservative cutoff should achieve.
False negative rate is about 22.7 percent: nearly a quarter of genuinely good applicants get rejected. That is the price paid for the conservative stance.

## Task 5: Model evaluation (out-of-sample)
- Apply the fitted model to the test set
- Compute out-of-sample ROC curve + AUC
- Compute confusion matrix on test set
- Compare in-sample vs out-of-sample AUC (overfitting check)

Step A: predict on the test set.

In [ ]:
p_test = model.predict_proba(features_test)[:, 1]

Step B: out-of-sample ROC and AUC.

In [ ]:
fpr_test, tpr_test, _ = roc_curve(target_test, p_test)
auc_test = auc(fpr_test, tpr_test)
auc_test

Step C: out-of-sample confusion matrix.

In [ ]:
x_bar_test = (p_test >= business_cutoff).astype(int)
cm_test = confusion_matrix(target_test, x_bar_test, normalize='all')
cm_test

Test AUC came out around 0.78 versus train AUC around 0.82, a drop of about 0.04. A small gap like this suggests the model generalizes reasonably well rather than overfitting.

AUC can be read as: take one random good applicant and one random bad applicant from the data. AUC is the probability that the model ranks the good applicant higher than the bad one.

AUC = 0.5 means the model is no better than a coin flip.
AUC = 1.0 means perfect separation between the two classes.
AUC = 0.78 (the test result here) means a random good/bad pair is ranked correctly about 78 percent of the time.

## Task 6: Model application
- Given a new applicant's raw data, preprocess it the same way
- Predict their probability of good/bad credit
- Apply the point cutoff to make a lending decision

In [ ]:
new_applicant_data = {
    'checking_status': '0<=X<200',
    'duration': 12,
    'credit_history': 'existing paid',
    'purpose': 'furniture/equipment',
    'credit_amount': 1845,
    'savings_status': '<100',
    'employment': '4<=X<7',
    'installment_commitment': 2,
    'personal_status': 'male div/sep',
    'other_parties': 'guarantor',
    'residence_since': 3,
    'property_magnitude': 'no known property',
    'age': 52,
    'other_payment_plans': 'none',
    'housing': 'own',
    'existing_credits': 1,
    'job': 'unskilled resident',
    'num_dependents': 1,
    'own_telephone': 'yes',
    'foreign_worker': 'yes',
}
new_applicant = DataFrame([new_applicant_data])
new_applicant = new_applicant.astype(credit_risk.iloc[:, :-1].dtypes.to_dict())

In [ ]:
z_cont_appl_raw = new_applicant.iloc[:, i_cont]
z_cat_appl_raw = new_applicant.iloc[:, i_cat]

z_cat_appl = enc.transform(z_cat_appl_raw)
z_cont_appl = quantile_transformer.transform(z_cont_appl_raw)

z_appl = concatenate((z_cat_appl, z_cont_appl), axis=1)

In [ ]:
p_appl = model.predict_proba(z_appl)[:, 1]
x_bar_appl = (p_appl >= business_cutoff).astype(int)

print('Predicted class =', 'good' if x_bar_appl == 1 else 'bad')
print('Probability of being a good credit risk =', p_appl.round(3)[0])